# 03_scrapper.py

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time

# --- 1. Variables básicas ---
usuario = "SilentStorm63"
base_url = f"https://www.backloggd.com/u/{usuario}/games/?page="
pagina = 1

todos_los_juegos = []
urls_existentes = set()

headers = {
    "User-Agent": "Mozilla/5.0"
}

# --- 2. Loop para recorrer páginas ---
while True:
    print(f"Scrapeando página {pagina}...")
    
    url = base_url + str(pagina)
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"⚠️ Error al cargar página {pagina}. Código {response.status_code}")
        break
    
    soup = BeautifulSoup(response.content, 'html.parser')
    
    bloques_juegos = soup.find_all('div', class_='rating-hover')

    juegos_nuevos_encontrados = 0  # contador de juegos nuevos

    for bloque in bloques_juegos:
        try:
            link_element = bloque.find('a', class_='cover-link')
            nombre_element = bloque.find('div', class_='game-text-centered')

            if link_element and nombre_element:
                url_relativa = link_element.get('href')
                url_juego = "https://www.backloggd.com" + url_relativa
                nombre_juego = nombre_element.get_text(strip=True)

                if url_juego not in urls_existentes:  # 🚨 solo si es realmente nuevo
                    todos_los_juegos.append({
                        "Juego": nombre_juego,
                        "URL": url_juego
                    })
                    urls_existentes.add(url_juego)
                    juegos_nuevos_encontrados += 1
        except Exception as e:
            print(f"⚠️ Error leyendo un juego: {e}")

    if juegos_nuevos_encontrados == 0:
        print(f"✅ No se encontraron juegos nuevos en página {pagina}. Fin del scraping.")
        break

    pagina += 1
    time.sleep(1)

# --- 3. Guardar en Excel ---
df = pd.DataFrame(todos_los_juegos)
df.to_excel("backloggd_juegos_completo.xlsx", index=False, engine='openpyxl')
print(f"✅ ¡Scraping completo! Juegos encontrados: {len(df)}")
print(df.head())

Scrapeando página 1...
Scrapeando página 2...
Scrapeando página 3...
Scrapeando página 4...
✅ No se encontraron juegos nuevos en página 4. Fin del scraping.
✅ ¡Scraping completo! Juegos encontrados: 91
                           Juego  \
0                  Marvel Rivals   
1  Max: The Curse of Brotherhood   
2                        Destiny   
3              Halo 5: Guardians   
4             Forza Motorsport 5   

                                                 URL  
0     https://www.backloggd.com/games/marvel-rivals/  
1  https://www.backloggd.com/games/max-the-curse-...  
2           https://www.backloggd.com/games/destiny/  
3  https://www.backloggd.com/games/halo-5-guardians/  
4  https://www.backloggd.com/games/forza-motorspo...  


# Extraccion de logs

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# --- 1. Cargar tu Excel de juegos ---
df = pd.read_excel('backloggd_juegos_completo.xlsx')

# --- 2. Configurar navegador ---
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # no abre ventana
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --- 3. Agregar nuevas columnas ---
df["Horas Jugadas"] = ""
df["Fecha Inicio"] = ""
df["Fecha Fin"] = ""

# --- 4. Extraer logs de cada juego ---
for index, row in df.iterrows():
    url = row['URL']
    print(f"⏳ Extrayendo logs de: {row['Juego']}")

    try:
        driver.get(url)
        time.sleep(3)  # esperar que cargue la página

        # Buscar elementos
        try:
            horas = driver.find_element(By.CSS_SELECTOR, ".col-auto.playtime").text.strip()
        except:
            horas = ""

        try:
            fechas = driver.find_elements(By.CSS_SELECTOR, ".game-log-date")
            fecha_inicio = fechas[0].text.strip() if len(fechas) > 0 else ""
            fecha_fin = fechas[1].text.strip() if len(fechas) > 1 else ""
        except:
            fecha_inicio = ""
            fecha_fin = ""

        # Actualizar DataFrame
        df.at[index, "Horas Jugadas"] = horas
        df.at[index, "Fecha Inicio"] = fecha_inicio
        df.at[index, "Fecha Fin"] = fecha_fin

    except Exception as e:
        print(f"⚠️ Error en {url}: {e}")

# --- 5. Guardar el nuevo Excel ---
df.to_excel("backloggd_juegos_logs.xlsx", index=False)
print("✅ ¡Excel actualizado con logs!")

# --- 6. Cerrar navegador ---
driver.quit()


⏳ Extrayendo logs de: Marvel Rivals
⏳ Extrayendo logs de: Max: The Curse of Brotherhood
⏳ Extrayendo logs de: Destiny
⏳ Extrayendo logs de: Halo 5: Guardians
⏳ Extrayendo logs de: Forza Motorsport 5
⏳ Extrayendo logs de: Grounded
⏳ Extrayendo logs de: Human: Fall Flat
⏳ Extrayendo logs de: Dead by Daylight
⏳ Extrayendo logs de: Phogs!
⏳ Extrayendo logs de: Halo Infinite
⏳ Extrayendo logs de: Spelunky 2
⏳ Extrayendo logs de: The Last of Us: Left Behind - Remastered
⏳ Extrayendo logs de: The Last of Us Part I
⏳ Extrayendo logs de: Fall Guys
⏳ Extrayendo logs de: Stories Untold
⏳ Extrayendo logs de: Fortnite
⏳ Extrayendo logs de: A Little to the Left
⏳ Extrayendo logs de: Lost in Random
⏳ Extrayendo logs de: What Remains of Edith Finch
⏳ Extrayendo logs de: Marvel's Spider-Man Remastered
⏳ Extrayendo logs de: Horizon Zero Dawn: Complete Edition
⏳ Extrayendo logs de: Core Keeper
⏳ Extrayendo logs de: Rise of the Tomb Raider: 20 Year Celebration
⏳ Extrayendo logs de: SSX 3
⏳ Extrayendo logs